# Differential Expression Analysis: HCC and iCCA

## Comprehensive Analysis with Statistical Testing and Disease-Specific Comparisons

This notebook performs:
- Quality control and preprocessing
- Differential expression analysis
- Statistical testing (t-test, Wilcoxon, DESeq2-like approaches)
- Visualization (volcano plots, heatmaps, MA plots)
- Disease-specific comparisons (HCC vs iCCA)
- Pathway and functional enrichment analysis

## 1. Import Libraries and Set Parameters

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import ttest_ind, mannwhitneyu, f_oneway
import warnings
warnings.filterwarnings('ignore')

# Plotting parameters
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 11

# Analysis parameters
FDR_THRESHOLD = 0.05
FC_THRESHOLD = 1.5  # log2 fold change threshold
P_VALUE_THRESHOLD = 0.05

## 2. Load and Explore Data

In [ ]:
# Load expression data and metadata
# Replace with your actual data sources

def load_expression_data(file_path):
    """
    Load gene expression data from CSV or HDF5 format
    Expected format: rows=genes, columns=samples
    """
    if file_path.endswith('.csv'):
        data = pd.read_csv(file_path, index_col=0)
    elif file_path.endswith('.h5'):
        data = pd.read_hdf(file_path)
    else:
        raise ValueError("Unsupported file format")
    return data

def load_metadata(file_path):
    """
    Load sample metadata
    Expected columns: sample_id, disease_type, condition, batch (optional)
    """
    metadata = pd.read_csv(file_path, index_col=0)
    return metadata

# Example: Load your data
# expression_data = load_expression_data('path/to/expression_matrix.csv')
# metadata = load_metadata('path/to/metadata.csv')

print("Data loading functions defined. Replace with your actual data paths.")

## 3. Quality Control and Preprocessing

In [ ]:
def perform_qc(expression_data, min_reads=1, min_genes_per_sample=200):
    """
    Perform quality control on expression data
    """
    print("Initial data shape:", expression_data.shape)
    
    # Remove genes with very low expression
    genes_to_keep = (expression_data > min_reads).sum(axis=1) > 0
    expression_data = expression_data[genes_to_keep]
    
    # Calculate QC metrics
    qc_metrics = pd.DataFrame({
        'total_reads': expression_data.sum(axis=0),
        'num_genes': (expression_data > 0).sum(axis=0),
        'median_reads_per_gene': expression_data.median(axis=0)
    })
    
    print("\nQC Metrics Summary:")
    print(qc_metrics.describe())
    
    return expression_data, qc_metrics

def normalize_data(expression_data, method='log2cpm'):
    """
    Normalize expression data
    Methods: 'log2cpm', 'tpm', 'quantile', 'rpkm'
    """
    if method == 'log2cpm':
        # CPM normalization followed by log2 transformation
        cpm = (expression_data + 1) / (expression_data.sum(axis=0) + 1) * 1e6
        normalized = np.log2(cpm + 1)
    elif method == 'quantile':
        # Quantile normalization
        normalized = expression_data.rank(method='average').subtract(
            expression_data.rank(method='average').min()).div(
            expression_data.rank(method='average').max() - 
            expression_data.rank(method='average').min())
    else:
        raise ValueError(f"Unknown normalization method: {method}")
    
    return normalized

print("QC and normalization functions defined.")

## 4. Differential Expression Analysis - Statistical Testing

In [ ]:
def perform_de_analysis(expression_data, metadata, group_col, condition1, condition2, 
                         test_method='ttest', batch_col=None):
    """
    Perform differential expression analysis between two conditions
    
    Parameters:
    - expression_data: gene x sample expression matrix
    - metadata: sample metadata with condition info
    - group_col: column name for grouping
    - condition1, condition2: two conditions to compare
    - test_method: 'ttest', 'wilcoxon', or 'anova'
    - batch_col: optional batch column for batch correction
    """
    
    # Get sample indices for each condition
    samples_cond1 = metadata[metadata[group_col] == condition1].index
    samples_cond2 = metadata[metadata[group_col] == condition2].index
    
    expr_cond1 = expression_data[samples_cond1]
    expr_cond2 = expression_data[samples_cond2]
    
    results = []
    
    for gene in expression_data.index:
        vals1 = expr_cond1.loc[gene].values
        vals2 = expr_cond2.loc[gene].values
        
        # Calculate means and fold change
        mean1 = np.mean(vals1)
        mean2 = np.mean(vals2)
        fc = mean2 - mean1  # log2 fold change
        
        # Statistical test
        if test_method == 'ttest':
            stat, pval = ttest_ind(vals1, vals2)
        elif test_method == 'wilcoxon':
            stat, pval = mannwhitneyu(vals1, vals2)
        elif test_method == 'anova':
            stat, pval = f_oneway(vals1, vals2)
        else:
            raise ValueError(f"Unknown test method: {test_method}")
        
        results.append({
            'gene': gene,
            'mean_cond1': mean1,
            'mean_cond2': mean2,
            'log2fc': fc,
            'pvalue': pval,
            'test_stat': stat
        })
    
    de_results = pd.DataFrame(results)
    
    # FDR correction (Benjamini-Hochberg)
    de_results['padj'] = stats.rankdata(de_results['pvalue']) / len(de_results) * \
                          (de_results['pvalue'].max() / de_results['pvalue'].max())
    from statsmodels.stats.multitest import multipletests
    rejected, padj, _, _ = multipletests(de_results['pvalue'], method='fdr_bh')
    de_results['padj'] = padj
    
    # Classify DE genes
    de_results['sig'] = ((de_results['padj'] < FDR_THRESHOLD) & 
                         (abs(de_results['log2fc']) > FC_THRESHOLD))
    de_results['direction'] = de_results['log2fc'].apply(
        lambda x: 'up' if x > FC_THRESHOLD else ('down' if x < -FC_THRESHOLD else 'ns')
    )
    
    return de_results.sort_values('padj')

print("DE analysis function defined.")

## 5. Visualization Functions

In [ ]:
def plot_volcano(de_results, title="Volcano Plot", save_path=None):
    """
    Create a volcano plot
    """
    fig, ax = plt.subplots(figsize=(10, 8))
    
    # Color points based on significance
    colors = ['red' if sig else 'gray' for sig in de_results['sig']]
    
    ax.scatter(de_results['log2fc'], -np.log10(de_results['padj']), 
              c=colors, alpha=0.6, s=50, edgecolors='black', linewidth=0.5)
    
    # Add threshold lines
    ax.axvline(FC_THRESHOLD, color='blue', linestyle='--', alpha=0.5, label=f'FC={FC_THRESHOLD}')
    ax.axvline(-FC_THRESHOLD, color='blue', linestyle='--', alpha=0.5)
    ax.axhline(-np.log10(FDR_THRESHOLD), color='green', linestyle='--', alpha=0.5, label=f'FDR={FDR_THRESHOLD}')
    
    ax.set_xlabel('log2 Fold Change', fontsize=12)
    ax.set_ylabel('-log10(adjusted p-value)', fontsize=12)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    
    return fig, ax

def plot_ma(de_results, title="MA Plot", save_path=None):
    """
    Create an MA plot (M vs A)
    """
    fig, ax = plt.subplots(figsize=(10, 8))
    
    # A = average log2 expression, M = log2 fold change
    de_results['A'] = (de_results['mean_cond1'] + de_results['mean_cond2']) / 2
    colors = ['red' if sig else 'gray' for sig in de_results['sig']]
    
    ax.scatter(de_results['A'], de_results['log2fc'], 
              c=colors, alpha=0.6, s=50, edgecolors='black', linewidth=0.5)
    
    ax.axhline(FC_THRESHOLD, color='blue', linestyle='--', alpha=0.5)
    ax.axhline(-FC_THRESHOLD, color='blue', linestyle='--', alpha=0.5)
    ax.axhline(0, color='black', linestyle='-', alpha=0.3)
    
    ax.set_xlabel('Average log2 Expression (A)', fontsize=12)
    ax.set_ylabel('log2 Fold Change (M)', fontsize=12)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    
    return fig, ax

def plot_heatmap(expression_data, metadata, de_results, top_n=20, group_col='disease_type', 
                  save_path=None):
    """
    Create a heatmap of top DE genes
    """
    top_genes = de_results.nlargest(top_n, 'padj' if de_results['padj'].min() < 1 else 'pvalue')['gene'].values
    
    expr_subset = expression_data.loc[top_genes]
    
    # Create column colors based on groups
    col_colors = pd.Categorical(metadata[group_col]).codes
    
    plt.figure(figsize=(12, 8))
    sns.clustermap(expr_subset, cmap='RdBu_r', center=0, 
                   col_colors=col_colors, figsize=(14, 8),
                   cbar_kws={'label': 'log2 Expression'},
                   yticklabels=True, xticklabels=False)
    
    plt.suptitle(f'Top {top_n} DE Genes Heatmap', fontsize=14, fontweight='bold', y=0.98)
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()

def plot_de_summary(de_results, title="DE Summary", save_path=None):
    """
    Create a summary plot of DE results
    """
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Distribution of log2 fold changes
    axes[0, 0].hist(de_results['log2fc'], bins=50, alpha=0.7, color='steelblue', edgecolor='black')
    axes[0, 0].set_xlabel('log2 Fold Change')
    axes[0, 0].set_ylabel('Count')
    axes[0, 0].set_title('Distribution of Fold Changes')
    axes[0, 0].grid(True, alpha=0.3)
    
    # Distribution of p-values
    axes[0, 1].hist(-np.log10(de_results['pvalue']), bins=50, alpha=0.7, color='darkorange', edgecolor='black')
    axes[0, 1].set_xlabel('-log10(p-value)')
    axes[0, 1].set_ylabel('Count')
    axes[0, 1].set_title('Distribution of p-values')
    axes[0, 1].grid(True, alpha=0.3)
    
    # DE counts by direction
    de_counts = de_results[de_results['sig']]['direction'].value_counts()
    axes[1, 0].bar(de_counts.index, de_counts.values, color=['red', 'blue'], alpha=0.7, edgecolor='black')
    axes[1, 0].set_ylabel('Count')
    axes[1, 0].set_title('DE Genes by Direction')
    for i, v in enumerate(de_counts.values):
        axes[1, 0].text(i, v + 1, str(v), ha='center', fontweight='bold')
    axes[1, 0].grid(True, alpha=0.3, axis='y')
    
    # Cumulative DE
    sorted_pvals = np.sort(de_results['padj'])
    axes[1, 1].plot(sorted_pvals)
    axes[1, 1].axhline(FDR_THRESHOLD, color='red', linestyle='--', label=f'FDR={FDR_THRESHOLD}')
    axes[1, 1].set_xlabel('Gene Rank')
    axes[1, 1].set_ylabel('Adjusted p-value')
    axes[1, 1].set_title('Cumulative DE Results')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.suptitle(title, fontsize=14, fontweight='bold', y=1.00)
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    
    return fig, axes

print("Visualization functions defined.")

## 6. Disease-Specific Comparisons (HCC vs iCCA)

In [ ]:
def compare_diseases(expression_data, metadata, disease1='HCC', disease2='iCCA'):
    """
    Compare gene expression between two diseases
    """
    
    print(f"\n{'='*60}")
    print(f"Disease Comparison: {disease1} vs {disease2}")
    print(f"{'='*60}")
    
    # Perform DE analysis
    disease_de = perform_de_analysis(expression_data, metadata, 
                                     group_col='disease_type',
                                     condition1=disease1,
                                     condition2=disease2,
                                     test_method='ttest')
    
    # Summary statistics
    n_de = disease_de['sig'].sum()
    n_up = (disease_de['direction'] == 'up').sum()
    n_down = (disease_de['direction'] == 'down').sum()
    
    print(f"\nDE Results:")
    print(f"  Total DE genes: {n_de}")
    print(f"  Upregulated in {disease2}: {n_up}")
    print(f"  Downregulated in {disease2}: {n_down}")
    
    print(f"\nTop 10 Upregulated Genes:")
    top_up = disease_de[disease_de['direction'] == 'up'].head(10)
    print(top_up[['gene', 'log2fc', 'padj']].to_string())
    
    print(f"\nTop 10 Downregulated Genes:")
    top_down = disease_de[disease_de['direction'] == 'down'].head(10)
    print(top_down[['gene', 'log2fc', 'padj']].to_string())
    
    return disease_de

def disease_comparison_heatmap(expression_data, metadata, de_results, disease1='HCC', disease2='iCCA',
                               top_n=20, save_path=None):
    """
    Create a disease-specific comparison heatmap
    """
    top_genes = de_results.head(top_n)['gene'].values
    
    expr_subset = expression_data.loc[top_genes]
    
    # Group samples by disease
    disease_colors = metadata['disease_type'].map({
        disease1: 0,
        disease2: 1
    })
    
    fig = plt.figure(figsize=(14, 8))
    sns.clustermap(expr_subset, cmap='RdBu_r', center=0,
                  col_colors=disease_colors, figsize=(14, 8),
                  cbar_kws={'label': 'log2 Expression'},
                  yticklabels=True, xticklabels=False)
    
    plt.suptitle(f'Top {top_n} DE Genes: {disease1} vs {disease2}', 
                fontsize=14, fontweight='bold', y=0.98)
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()

print("Disease comparison functions defined.")

## 7. Functional Enrichment Analysis

In [ ]:
def simple_enrichment_analysis(de_results, background_genes=None, top_n=50):
    """
    Perform simple functional enrichment analysis
    Note: For comprehensive analysis, use external tools like enrichR, g:Profiler, or GSEA
    """
    
    de_genes = de_results[de_results['sig']].head(top_n)['gene'].values
    
    # Example pathway database (simplified)
    pathway_db = {
        'Apoptosis': ['TP53', 'BAX', 'BAD', 'CASP3', 'CASP9'],
        'Cell Cycle': ['CDK1', 'CDK4', 'CCNB1', 'CCND1', 'RB1'],
        'DNA Repair': ['BRCA1', 'BRCA2', 'MLH1', 'MSH2', 'XPA'],
        'Angiogenesis': ['VEGFA', 'VEGFR2', 'FLK1', 'TEK', 'ANGPT1'],
        'Immune Response': ['CD8A', 'CD4', 'IL2', 'TNF', 'IFNG'],
        'Metabolism': ['HK2', 'PKM', 'LDHA', 'PFKFB3', 'SLC2A1'],
        'EMT': ['CDH1', 'CDH2', 'VIM', 'SNAIL', 'ZEB1'],
        'Wnt Signaling': ['WNT1', 'GSK3B', 'APC', 'TCF4', 'LEF1'],
        'MAPK Pathway': ['RAF1', 'MEK1', 'ERK1', 'ERK2', 'EGFR'],
        'PI3K/AKT': ['PIK3CA', 'AKT1', 'PTEN', 'mTOR', 'GSK3B']
    }
    
    enrichment_results = []
    
    for pathway, genes in pathway_db.items():
        overlap = set(de_genes) & set(genes)
        if len(overlap) > 0:
            enrichment_results.append({
                'pathway': pathway,
                'overlap_count': len(overlap),
                'pathway_size': len(genes),
                'overlapping_genes': ','.join(overlap),
                'enrichment_ratio': len(overlap) / len(genes)
            })
    
    enrichment_df = pd.DataFrame(enrichment_results).sort_values('overlap_count', ascending=False)
    
    return enrichment_df

def plot_enrichment(enrichment_df, title="Pathway Enrichment", save_path=None):
    """
    Plot enrichment analysis results
    """
    fig, ax = plt.subplots(figsize=(10, 8))
    
    colors = plt.cm.viridis(np.linspace(0.3, 0.9, len(enrichment_df)))
    bars = ax.barh(enrichment_df['pathway'], enrichment_df['overlap_count'], color=colors, edgecolor='black')
    
    ax.set_xlabel('Number of Overlapping Genes', fontsize=12)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='x')
    
    # Add value labels
    for bar in bars:
        width = bar.get_width()
        ax.text(width, bar.get_y() + bar.get_height()/2, f'{int(width)}',
               ha='left', va='center', fontweight='bold')
    
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    
    return fig, ax

print("Enrichment analysis functions defined.")

## 8. Main Analysis Workflow

In [ ]:
# WORKFLOW: Execute this section to run complete analysis

# Step 1: Load data
# expression_data = load_expression_data('path/to/expression_data.csv')
# metadata = load_metadata('path/to/metadata.csv')

# Step 2: QC and preprocessing
# expression_data, qc_metrics = perform_qc(expression_data)

# Step 3: Normalize
# expression_normalized = normalize_data(expression_data, method='log2cpm')

# Step 4: HCC vs Normal
# de_hcc_normal = perform_de_analysis(expression_normalized, metadata, 
#                                     group_col='disease_type',
#                                     condition1='HCC', 
#                                     condition2='Normal',
#                                     test_method='ttest')

# Step 5: iCCA vs Normal
# de_icca_normal = perform_de_analysis(expression_normalized, metadata,
#                                      group_col='disease_type',
#                                      condition1='iCCA',
#                                      condition2='Normal',
#                                      test_method='ttest')

# Step 6: HCC vs iCCA
# de_hcc_icca = compare_diseases(expression_normalized, metadata, 'HCC', 'iCCA')

# Step 7: Visualization
# plot_volcano(de_hcc_normal, title='HCC vs Normal - Volcano Plot')
# plot_volcano(de_icca_normal, title='iCCA vs Normal - Volcano Plot')
# plot_volcano(de_hcc_icca, title='HCC vs iCCA - Volcano Plot')

# plot_ma(de_hcc_normal, title='HCC vs Normal - MA Plot')
# plot_de_summary(de_hcc_normal, title='HCC vs Normal - Summary')

# Step 8: Enrichment analysis
# enrichment_hcc = simple_enrichment_analysis(de_hcc_normal)
# enrichment_icca = simple_enrichment_analysis(de_icca_normal)
# plot_enrichment(enrichment_hcc, title='HCC vs Normal - Enriched Pathways')
# plot_enrichment(enrichment_icca, title='iCCA vs Normal - Enriched Pathways')

print("\nWorkflow template ready. Uncomment and execute sections with your data.")

## 9. Export Results

In [ ]:
def export_results(de_results, filename='de_results.csv'):
    """
    Export DE results to CSV
    """
    de_results.to_csv(filename, index=False)
    print(f"Results exported to {filename}")

def export_sig_genes(de_results, filename_up='upregulated_genes.txt', filename_down='downregulated_genes.txt'):
    """
    Export significantly DE genes to text files
    """
    up_genes = de_results[de_results['direction'] == 'up']['gene'].values
    down_genes = de_results[de_results['direction'] == 'down']['gene'].values
    
    with open(filename_up, 'w') as f:
        f.write('\n'.join(up_genes))
    
    with open(filename_down, 'w') as f:
        f.write('\n'.join(down_genes))
    
    print(f"Upregulated genes exported to {filename_up}")
    print(f"Downregulated genes exported to {filename_down}")

# Example usage:
# export_results(de_hcc_normal, 'hcc_vs_normal_de_results.csv')
# export_sig_genes(de_hcc_normal, 'hcc_upregulated.txt', 'hcc_downregulated.txt')

print("Export functions defined.")

## 10. Session Information and Package Versions

In [ ]:
import sys

print("Python Version:", sys.version)
print("\nKey Package Versions:")
packages = ['pandas', 'numpy', 'matplotlib', 'seaborn', 'scipy', 'statsmodels']
for pkg in packages:
    try:
        exec(f"import {pkg}")
        exec(f"print(f'{pkg}: {{pkg.__version__}}')".replace('{{', '{').replace('}}', '}'))
    except:
        print(f"{pkg}: Not installed")

## Notes and References

### Data Requirements:
- **Expression Matrix**: Genes (rows) × Samples (columns), can be raw counts or normalized values
- **Metadata**: Sample information with at least:
  - sample_id
  - disease_type (HCC, iCCA, Normal, etc.)
  - Optional: batch, condition, age, stage, etc.

### Statistical Methods:
- **t-test**: For normally distributed data with two groups
- **Mann-Whitney U (Wilcoxon)**: Non-parametric alternative to t-test
- **ANOVA**: For multiple groups
- **FDR Correction**: Benjamini-Hochberg method (recommended)

### Thresholds:
- **log2 Fold Change**: ±1.5 (default, adjustable)
- **Adjusted p-value (FDR)**: 0.05 (default, adjustable)

### For Further Analysis:
1. **GSEA/Pathway Analysis**: Use MSigDB, KEGG, or Reactome
2. **Network Analysis**: STRING, Cytoscape
3. **Machine Learning**: Classification, clustering, survival analysis
4. **Integration**: Multi-omics analysis (proteomics, methylation, CNV)

### References:
- Ritchie, M. E., et al. (2015). limma powers differential expression analyses for RNA-sequencing and microarray studies. Nucleic Acids Research.
- Love, M. I., et al. (2014). Moderated estimation of fold change and dispersion for RNA-seq data with DESeq2. Genome Biology.
- Robinson, M. D., et al. (2010). edgeR: differential expression analysis of digital gene expression data. Bioinformatics.